# 03 - Full training on Kaggle (2x T4)

Setup (once):
1. Upload the code as a **private** Kaggle dataset `dataslayer-src` (only `src/`, `configs/`, `requirements.txt`;
   never the competition data): from the repo root run
   `kaggle datasets create -p kaggle_upload --dir-mode zip` (see `README.md`).
2. New notebook -> add inputs: competition `data-slayer-4` + dataset `dataslayer-src`.
3. Settings: Accelerator **GPU T4 x2**, Internet **on** (pretrained weights from timm/HF hub).

Kaggle images already ship torch/timm/accelerate, so nothing is reinstalled.
Pipeline: prepare (OpenCV test-style frame cache, ~15 min) -> 5-fold CV with `accelerate launch` (DDP over both T4s,
fp16, per-epoch metrics) -> fold-ensemble inference -> `/kaggle/working/submission.csv`.

In [ ]:
import os
import subprocess
from pathlib import Path

COMP = next(Path("/kaggle/input").rglob("sample_submission.csv")).parent
CODE = next(Path("/kaggle/input").rglob("src/core/config.py")).parents[2]
os.environ["PYTHONPATH"] = str(CODE / "src")
os.environ["TQDM_MININTERVAL"] = "60"  # keep committed-run logs small
CONFIG = str(CODE / "configs" / "base.yaml")
OVERRIDES = [f"paths.data_dir={COMP}"]  # e.g. add "train.optimizer=muon", "name=convnext_t_muon"
SET = " ".join(OVERRIDES)
print(COMP, CODE)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, timm, accelerate; print(torch.__version__, timm.__version__, accelerate.__version__)"

In [ ]:
!python -m cli.prepare --config {CONFIG} --set {SET} --workers 4

In [ ]:
!accelerate launch --multi_gpu --num_processes 2 --mixed_precision fp16 -m cli.train --config {CONFIG} --set {SET}

In [ ]:
!accelerate launch --multi_gpu --num_processes 2 --mixed_precision fp16 -m cli.predict --config {CONFIG} --set {SET}

In [ ]:
import json
import shutil

import pandas as pd

import sys
sys.path.insert(0, os.environ["PYTHONPATH"])
from core.config import load_config

cfg = load_config(CONFIG, OVERRIDES)
print(json.dumps(json.loads((cfg.run_dir / "cv.json").read_text()), indent=1))
for h in sorted(cfg.run_dir.glob("fold*/history.csv")):
    print(h.parent.name); display(pd.read_csv(h).round(4))
shutil.copy(cfg.run_dir / "submission.csv", "/kaggle/working/submission.csv")